# Household WASH estimates for Nigerian states and the FCT

This notebook estimates basic drinking water, sanitation and hygiene for 2013, 2018 and 2024. It also estimates the 2024 minus 2013 change in percentage points.

Put this notebook beside `Nigeria_DHS_2013_2018_2024_Households_Climate.csv.gz`. Open it in JupyterLab and choose **Run → Run All Cells**. If the Python packages are missing, run `%pip install pandas numpy scipy matplotlib python-docx pillow` in a code cell and restart the kernel.

For automatic **Word 97–2003 (.doc)** export on Windows, install [LibreOffice Writer](https://www.libreoffice.org/download/download-libreoffice/) in its default location. If you installed it elsewhere, set `LIBREOFFICE_PATH` to the full path of `soffice.com` in a code cell before exporting. The notebook also writes an editable **.docx** that you can open in desktop Word and save as **Word 97–2003 Document (*.doc)** if the automatic converter is unavailable.

The source file stays compressed. The notebook uses all households with a known state and retains households with missing GPS or climate data in the descriptive analysis.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit, logit
from scipy.stats import t

try:
    from google.colab import files
except ModuleNotFoundError:
    files = None

FILENAME = "Nigeria_DHS_2013_2018_2024_Households_Climate.csv.gz"
INPUT = Path("/content" if files else ".") / FILENAME
OUTPUT = INPUT.parent / "wash_state_outputs"

if not INPUT.is_file() and files is not None:
    print(f"Choose {FILENAME}", flush=True)
    partial = INPUT.with_suffix(".gz.part")
    try:
        files.upload_file(str(partial))
        if partial.is_file() and partial.stat().st_size:
            partial.replace(INPUT)
    finally:
        partial.unlink(missing_ok=True)
if not INPUT.is_file():
    raise FileNotFoundError(f"Put {FILENAME} beside the notebook and rerun.")

KEY = ["survey_round", "hv001", "hv002"]
STRATUM = "stratum_hv023_key"
PSU = "psu_key"
DOMAINS = ["water_basic", "sanitation_basic", "hygiene_basic"]
REQUIRED = [*KEY, STRATUM, PSU, "state", "hh_weight", "hv005", *DOMAINS]
d = pd.read_csv(INPUT, usecols=REQUIRED, compression="gzip", low_memory=False)
expected = {2013: 38522, 2018: 40427, 2024: 40047}
if d.groupby("survey_round").size().to_dict() != expected:
    raise ValueError("Unexpected household totals. Check that you selected the combined household file.")
if d[KEY].isna().any().any() or d.duplicated(KEY).any():
    raise ValueError("Missing or duplicate household IDs.")
if d[[STRATUM, PSU]].isna().any().any():
    raise ValueError("A record is missing its survey stratum or PSU.")
states = sorted(d.state.dropna().unique())
if d.state.isna().any() or len(states) != 37 or "FCT" not in states:
    raise ValueError("Expected 36 states and the FCT with no missing state labels.")
for year, frame in d.groupby("survey_round"):
    if set(frame.state) != set(states):
        raise ValueError(f"{year}: state labels do not match the other rounds.")
if not np.isfinite(d.hh_weight).all() or not d.hh_weight.gt(0).all():
    raise ValueError("Household weights must be positive and finite.")
if not np.allclose(d.hh_weight, d.hv005 / 1_000_000, rtol=0, atol=1e-12):
    raise ValueError("hh_weight does not match the scaled DHS household weight hv005.")
for column in DOMAINS:
    if not d[column].dropna().isin([0, 1]).all():
        raise ValueError(f"{column} must contain only 0, 1 or a blank.")
print(f"Loaded {len(d):,} households.")


## 1. Estimate prevalence and change

The weighted estimate uses households with a known outcome for the selected state and indicator. The notebook uses `hh_weight`, recorded `hv023` strata and `hv021` PSUs. First-stage Taylor linearisation retains the full round design for state domains. Prevalence intervals use the logit scale. A single-PSU stratum with a known outcome gets the average variance contribution of the state's active multi-PSU strata. If there is no eligible donor stratum or the estimate is on the 0% or 100% boundary, the interval is blank.

Change equals the 2024 estimate minus the 2013 estimate. Its approximate 95% interval combines the two standard errors assuming independent rounds, using the smaller round-level design degrees of freedom. It describes differences between two household survey estimates.

Review flags use study screening rules. A row is flagged if it has fewer than **50** known outcomes, fewer than **5** PSUs with known outcomes, outcome coverage below **80%** after weighting, a prevalence interval wider than **30 percentage points**, or an unavailable interval. A change is flagged when either endpoint is flagged. Hygiene estimates refer to households with known hygiene status.


In [ ]:
rounds = dict(tuple(d.groupby("survey_round", sort=True)))
design = {}
design_rows = []
for year, frame in rounds.items():
    mapping = frame[[STRATUM, PSU]].drop_duplicates()
    if mapping[PSU].duplicated().any():
        raise ValueError(f"{year}: a PSU appears in more than one survey stratum.")
    sizes = mapping.groupby(STRATUM).size()
    df = len(mapping) - len(sizes)
    if df <= 0:
        raise ValueError(f"{year}: insufficient survey design degrees of freedom.")
    design[year] = (mapping, sizes, df)
    design_rows.append({
        "survey_round": year, "strata": len(sizes), "psus": len(mapping),
        "single_psu_strata": int(sizes.eq(1).sum()), "design_df": df,
    })

def estimate(year, state, outcome):
    frame = rounds[year]
    mapping, sizes, df = design[year]
    domain = frame.state.eq(state)
    known = domain & frame[outcome].notna()
    weights = frame.hh_weight
    count = int(domain.sum())
    n_known = int(known.sum())
    denominator = float(weights[known].sum())
    record = {
        "survey_round": year, "state": state, "indicator": outcome,
        "households_in_group": count, "unweighted_denominator": n_known,
        "unweighted_missing": count - n_known,
        "weighted_outcome_known_pct": 100 * denominator / weights[domain].sum() if count else np.nan,
        "known_psus": int(frame.loc[known, PSU].nunique()),
        "weighted_pct": np.nan, "se_pct": np.nan, "ci95_low": np.nan, "ci95_high": np.nan,
        "design_df": df, "single_psu_strata_adjusted": 0,
    }
    if not n_known:
        return record

    p = float((weights[known] * frame.loc[known, outcome]).sum() / denominator)
    record["weighted_pct"] = 100 * p
    if not 0 < p < 1:
        return record  # A logit interval is undefined at the boundaries.

    # Sum the linearised ratio residuals within each sampled PSU.
    residual = (weights[known] * (frame.loc[known, outcome] - p)).rename("residual")
    psu_totals = (
        frame.loc[known, [STRATUM, PSU]].join(residual)
        .groupby([STRATUM, PSU]).residual.sum()
        .reindex(pd.MultiIndex.from_frame(mapping), fill_value=0.0)
    )
    centred = psu_totals - psu_totals.groupby(level=0).transform("mean")
    multi = sizes[sizes.gt(1)]
    variance_by_stratum = centred.pow(2).groupby(level=0).sum().loc[multi.index]
    variance_by_stratum *= multi / (multi - 1)

    active = pd.Index(frame.loc[known, STRATUM].unique())
    lonely = int(sizes.loc[active].eq(1).sum())
    variance = float(variance_by_stratum.sum())
    if lonely:
        comparable = variance_by_stratum.loc[variance_by_stratum.index.intersection(active)]
        if comparable.empty:
            return record
        variance += lonely * float(comparable.mean())
        record["single_psu_strata_adjusted"] = lonely

    se = np.sqrt(variance) / denominator
    record["se_pct"] = 100 * se
    margin = t.ppf(0.975, df) * se / (p * (1 - p))
    record["ci95_low"], record["ci95_high"] = 100 * expit([logit(p) - margin, logit(p) + margin])
    return record


prevalence = pd.DataFrame([
    estimate(year, state, outcome)
    for year in rounds for state in states for outcome in DOMAINS
])

def review_flags(row):
    checks = [
        (row.unweighted_denominator < 50, "fewer_than_50_known"),
        (row.known_psus < 5, "fewer_than_5_psus"),
        (row.weighted_outcome_known_pct < 80, "coverage_below_80pct"),
        (row.ci95_high - row.ci95_low > 30, "ci_wider_than_30pp"),
        (pd.isna(row.ci95_low) or pd.isna(row.ci95_high), "ci_unavailable"),
    ]
    return " | ".join(label for flag, label in checks if flag)

prevalence["review_flags"] = prevalence.apply(review_flags, axis=1)
index = ["state", "indicator"]
start = prevalence.loc[prevalence.survey_round.eq(2013)].set_index(index)
end = prevalence.loc[prevalence.survey_round.eq(2024)].set_index(index)
change = pd.DataFrame(index=start.index)
change["change_2024_minus_2013_pp"] = end.weighted_pct - start.weighted_pct
change["change_se_pp"] = np.hypot(start.se_pct, end.se_pct)
margin = t.ppf(0.975, min(design[2013][2], design[2024][2])) * change.change_se_pp
change["ci95_low_pp"] = change.change_2024_minus_2013_pp - margin
change["ci95_high_pp"] = change.change_2024_minus_2013_pp + margin
change["flags_2013"] = start.review_flags
change["flags_2024"] = end.review_flags
change["review_flag"] = change.flags_2013.ne("") | change.flags_2024.ne("")
change = change.reset_index()

assert len(prevalence) == 333 and not prevalence.duplicated(["survey_round", *index]).any()
assert len(change) == 111 and not change.duplicated(index).any()
assert (prevalence.unweighted_denominator + prevalence.unweighted_missing).eq(prevalence.households_in_group).all()

OUTPUT.mkdir(parents=True, exist_ok=True)
prevalence.to_csv(OUTPUT / "State_WASH_Prevalence.csv", index=False)
change.to_csv(OUTPUT / "State_WASH_Change_2013_2024.csv", index=False)
display(prevalence.loc[prevalence.state.eq("Benue")].round(2))
display(change.loc[change.state.eq("Benue")].round(2))
print(f"Saved {len(prevalence)} prevalence estimates and {len(change)} changes.")


## 2. Save figures and the Word report

The three figures show 2024 minus 2013 differences with approximate 95% confidence intervals. The open symbols flag estimates for review. Each JPEG is saved at **600 dpi**. The report includes all 37 areas and uses a dagger to mark estimates for review. The CSV files give the detailed denominators and reason for each flag.

Outputs are in `wash_state_outputs`. Begin with `State_WASH_Prevalence.csv`, filter to one state, and check that you see three indicators per round. Use the hygiene coverage column before interpreting hygiene change. These descriptive results precede formal hotspot tests.

References for the survey design and variance method include the [Guide to DHS Statistics](https://dhsprogram.com/pubs/pdf/DHSG1/Guide_to_DHS_Statistics_DHS-8.pdf) and the [survey package documentation](https://cran.r-project.org/web/packages/survey/survey.pdf).


In [ ]:
import os
import shutil
import subprocess
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
from docx import Document
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor
from PIL import Image


names = dict(zip(DOMAINS, ["Drinking water", "Sanitation", "Hygiene"]))
figure_width = 6.8
dpi = 600

def save_figure(figure, filename):
    path = OUTPUT / filename
    figure.savefig(path, format="jpeg", dpi=dpi, facecolor="white",
                   pil_kwargs={"quality": 95, "subsampling": 0})
    plt.close(figure)
    with Image.open(path) as image:
        if image.format != "JPEG" or image.info.get("dpi") != (dpi, dpi):
            raise ValueError(f"Could not verify JPEG resolution for {path.name}.")
    return path


figures = {}
for indicator in DOMAINS:
    plot = change.loc[change.indicator.eq(indicator)].sort_values("state", ascending=False)
    fig, ax = plt.subplots(figsize=(figure_width, 8.0), layout="constrained")
    fig.suptitle(f"Change in basic {names[indicator].lower()} by state", fontsize=13, fontweight="bold")
    pos = np.arange(len(plot))
    values = plot.change_2024_minus_2013_pp.to_numpy()
    low, high = plot.ci95_low_pp.to_numpy(), plot.ci95_high_pp.to_numpy()
    has_ci = np.isfinite(low) & np.isfinite(high)
    flagged = plot.review_flag.to_numpy()
    ax.axvline(0, color="#495663", linewidth=0.9, linestyle="--")
    ax.hlines(pos[has_ci], low[has_ci], high[has_ci], color="#7D8B96", linewidth=0.85)
    ax.scatter(values[~flagged], pos[~flagged], s=19, color="#176C87", zorder=3, label="No review flag")
    ax.scatter(values[flagged], pos[flagged], s=24, facecolors="white", edgecolors="#A34D20",
               zorder=3, label="Flagged for review")
    ax.set(yticks=pos, yticklabels=plot.state, ylim=(-0.7, len(plot) - 0.3),
           xlabel="2024 minus 2013 (percentage points)")
    ax.tick_params(axis="both", labelsize=8)
    ax.grid(axis="x", color="#E1E6EA", linewidth=0.65)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False, fontsize=8)
    figures[indicator] = save_figure(fig, f"State_Change_{indicator}_600dpi.jpg")

doc = Document()
doc.core_properties.title = "Household WASH access by state in Nigeria"
doc.core_properties.author = ""
section = doc.sections[0]
section.page_width, section.page_height = Inches(8.5), Inches(11)
section.left_margin = section.right_margin = Inches(0.85)
section.top_margin, section.bottom_margin = Inches(0.7), Inches(0.7)

for style_name, size in [("Normal", 10.5), ("Title", 25), ("Heading 1", 17), ("Heading 2", 13), ("Caption", 9)]:
    style = doc.styles[style_name]
    style.font.name = "Calibri"
    style.font.size = Pt(size)
    style.font.color.rgb = RGBColor.from_string("000000")
    style.paragraph_format.space_before = Pt(5)
    style.paragraph_format.space_after = Pt(5)
    if style_name == "Title":
        border = style.element.get_or_add_pPr().find(qn("w:pBdr"))
        if border is not None:
            border.getparent().remove(border)

def xml(parent, tag, **attributes):
    element = OxmlElement(f"w:{tag}")
    for key, value in attributes.items():
        element.set(qn(f"w:{key}"), str(value))
    parent.append(element)
    return element

def add_table(caption, headers, rows, widths, text_columns, font_size=9.5):
    doc.add_paragraph(caption, style="Caption").paragraph_format.keep_with_next = True
    table = doc.add_table(rows=0, cols=len(headers))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = False
    for column, width in zip(table.columns, widths):
        column.width = Inches(width)
    props = table._tbl.tblPr
    borders = xml(props, "tblBorders")
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        xml(borders, edge, val="single", sz=4, color="D9D9D9")
    margins = xml(props, "tblCellMar")
    for edge, value in (("top", 16), ("bottom", 16), ("left", 85), ("right", 85)):
        xml(margins, edge, w=value, type="dxa")

    for i, values in enumerate([headers, *rows]):
        row = table.add_row()
        row_props = row._tr.get_or_add_trPr()
        xml(row_props, "cantSplit")
        if i == 0:
            xml(row_props, "tblHeader")
        for j, (cell, value, width) in enumerate(zip(row.cells, values, widths)):
            cell.width = Inches(width)
            cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
            if i == 0 or i % 2 == 0:
                xml(cell._tc.get_or_add_tcPr(), "shd", fill="203A50" if i == 0 else "F2F6F9")
            paragraph = cell.paragraphs[0]
            paragraph.alignment = WD_ALIGN_PARAGRAPH.LEFT if j in text_columns else WD_ALIGN_PARAGRAPH.CENTER
            paragraph.paragraph_format.space_after = Pt(0)
            paragraph.paragraph_format.space_before = Pt(0)
            paragraph.paragraph_format.line_spacing = 1.05
            run = paragraph.add_run(str(value))
            run.bold = i == 0
            run.font.size = Pt(font_size)
            if i == 0:
                run.font.color.rgb = RGBColor.from_string("FFFFFF")
    return table

def add_figure(path, caption, description):
    paragraph = doc.add_paragraph()
    paragraph.paragraph_format.space_before = Pt(7)
    paragraph.paragraph_format.keep_with_next = True
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    shape = paragraph.add_run().add_picture(str(path), width=Inches(figure_width))
    shape._inline.docPr.set("descr", description)
    doc.add_paragraph(caption, style="Caption")

def confidence(row):
    if pd.isna(row.ci95_low) or pd.isna(row.ci95_high):
        return "—"
    return f"{row.ci95_low:.1f}–{row.ci95_high:.1f}"


def display_value(value):
    return f"{value:.1f}" if pd.notna(value) else "—"

def prevalence_entry(row):
    flag = "†" if row.review_flags else ""
    return f"{display_value(row.weighted_pct)}{flag} ({confidence(row)})"

prevalence_index = prevalence.set_index(["indicator", "state", "survey_round"])
change_index = change.set_index(["indicator", "state"])
doc.add_paragraph("Household WASH access by state in Nigeria", style="Title")
doc.add_paragraph("Nigeria Demographic and Health Survey rounds 2013, 2018 and 2024")
doc.add_paragraph(
    f"The analysis covers {len(d):,} households across 36 states and the Federal Capital Territory. "
    "Estimates describe households with known outcomes for the corresponding WASH indicator."
)
for number, indicator in enumerate(DOMAINS, start=1):
    heading = doc.add_heading(names[indicator], level=1)
    heading.paragraph_format.page_break_before = number > 1
    rows = []
    for state in states:
        rounds_for_state = [prevalence_index.loc[(indicator, state, year)] for year in (2013, 2018, 2024)]
        delta = change_index.loc[(indicator, state)]
        flag = "†" if delta.review_flag else ""
        delta_ci = "—" if pd.isna(delta.ci95_low_pp) else f"{delta.ci95_low_pp:.1f} to {delta.ci95_high_pp:.1f}"
        rows.append([state, *map(prevalence_entry, rounds_for_state),
                     f"{display_value(delta.change_2024_minus_2013_pp)}{flag} ({delta_ci})"])
    add_table(f"Table {number}. Weighted prevalence (%) and change (percentage points), with 95% confidence intervals",
              ["State or FCT", "2013", "2018", "2024", "2024 minus 2013"],
              rows, [1.1, 1.3, 1.3, 1.3, 1.8], {0}, font_size=8.5)
    doc.add_paragraph("† Review survey support, outcome coverage or interval precision in the CSV. A dash marks an unavailable interval.", style="Caption")

for number, indicator in enumerate(DOMAINS, start=1):
    heading = doc.add_heading(f"Change in basic {names[indicator].lower()}", level=1)
    heading.paragraph_format.page_break_before = True
    add_figure(figures[indicator], f"Figure {number}. Estimated change with approximate 95% confidence intervals.",
               f"Dot and interval plot of state changes in {names[indicator].lower()} between 2013 and 2024.")

doc.add_heading("Methods and interpretation", level=1).paragraph_format.page_break_before = True
doc.add_paragraph(
    "Estimates use the DHS household weight and the recorded survey strata and primary sampling units. "
    "Confidence intervals use first-stage Taylor linearisation. The variance for a contributing single-PSU stratum "
    "uses the mean contribution from eligible multi-PSU strata in that state domain."
)
doc.add_paragraph(
    "Change intervals combine uncertainty from two independent survey rounds. Review flags mark fewer than 50 "
    "known outcomes, fewer than five PSUs with known outcomes, weighted coverage below 80%, a prevalence interval "
    "wider than 30 percentage points, or an unavailable interval. The CSV gives known and missing counts, coverage, "
    "and the reason for each flag."
)
doc.add_paragraph(
    "Hygiene estimates use households with known hygiene status. Differences in outcome coverage across states "
    "and rounds can affect comparisons. Review coverage before interpreting the changes."
)

def find_soffice():
    custom = os.environ.get("LIBREOFFICE_PATH")
    if custom and Path(custom).is_file():
        return custom
    candidates = [shutil.which("soffice"), shutil.which("libreoffice"),
                  "/Applications/LibreOffice.app/Contents/MacOS/soffice"]
    for variable in ("ProgramFiles", "ProgramFiles(x86)"):
        if base := os.environ.get(variable):
            candidates.append(str(Path(base) / "LibreOffice" / "program" / "soffice.com"))
    return next((candidate for candidate in candidates if candidate and Path(candidate).is_file()), None)


docx_path = OUTPUT / "Nigeria_DHS_State_WASH_Report.docx"
doc.save(docx_path)

def export_legacy_word(source, soffice):
    with TemporaryDirectory() as scratch:
        temp = Path(scratch).resolve()
        result = subprocess.run(
            [soffice, f"-env:UserInstallation={(temp / 'profile').as_uri()}", "--headless",
             "--convert-to", "doc:MS Word 97", "--outdir", str(temp), str(source.resolve())],
            capture_output=True, text=True, timeout=180,
        )
        converted = temp / source.with_suffix(".doc").name
        if result.returncode or not converted.is_file():
            raise RuntimeError(result.stderr or result.stdout or "Converter returned no file.")
        with converted.open("rb") as stream:
            if stream.read(8) != bytes.fromhex("D0CF11E0A1B11AE1"):
                raise ValueError("The converter did not create a binary Word 97–2003 document.")
        shutil.copyfile(converted, source.with_suffix(".doc"))

soffice = find_soffice()
if soffice:
    try:
        export_legacy_word(docx_path, soffice)
        print(f"Saved {docx_path.with_suffix('.doc').resolve()}")
    except (RuntimeError, ValueError, OSError, subprocess.TimeoutExpired) as error:
        print(f"Word .doc conversion failed: {error}")
        print("Open the saved .docx in desktop Word and save it as Word 97–2003 Document (*.doc).")
else:
    print("LibreOffice was unavailable. Open the saved .docx in desktop Word and save it as Word 97–2003 Document (*.doc).")
print(f"Saved the report and three verified 600 dpi JPEG figures in {OUTPUT.resolve()}.")
